In [26]:
"""
Reddit Comment Labeling Pipeline (LM Studio, OpenAI-compatible API)
-------------------------------------------------------------------
- Input: dictionaries (Reddit-style with keys 'id' and 'body'), OR a dict mapping id->text,
         OR a pandas DataFrame with columns ['id','body'].
- Retries: Up to 3 attempts per (comment, task) with exponential backoff.
- Strict JSON validation against task schemas; otherwise retry.
- Output: NDJSON (one line per (comment_id, task)).
- Tasks implemented:
    - stance_intensity
    - epistemic_modality
    - justification_density
    - responsiveness (will ABSTAIN because no parent text provided)
    - civility

Requirements:
    pip install requests pandas
    reddit data has to be stored locally in a folder called 'data' (optional demo below).

LM Studio:
    - Enable the local HTTP server in LM Studio (OpenAI-compatible API).
    - Default endpoint: http://localhost:1234/v1
    - Set MODEL_NAME to the exact local model identifier shown in LM Studio.
"""

import json
import time
from typing import Dict, Any, List, Optional, Iterable, Tuple, Union
import requests
import pandas as pd
from pathlib import Path

In [27]:

# ------------------------
# Configuration
# ------------------------

LMSTUDIO_BASE_URL = "http://localhost:1234"   # Change if LM Studio runs on a different host/port
MODEL_NAME = "meta-llama-3.1-8b-instruct"        # e.g., "qwen2.5-7b-instruct"
TIMEOUT_SECONDS = 60                              # HTTP request timeout
MAX_RETRIES = 3                                   # Max attempts per (comment, task)
RETRY_BACKOFF_SECONDS = 1.5                       # Exponential backoff base

# ------------------------
# (Optional) Demo: Read reddit data
# ------------------------

In [28]:
p_sub = Path("data/submissions.ndjson")
p_com = Path("data/comments.ndjson")

print("Exists submissions:", p_sub.exists(), p_sub.resolve())
print("Exists comments   :", p_com.exists(), p_com.resolve())

df_submissions = pd.read_json(p_sub, lines=True)                    # <-- lines=True hier
df_comments    = pd.read_json(p_com, lines=True)

# Minimal subset for our pipeline
df_comments[["id", "body"]].dropna(subset=["id", "body"])

Exists submissions: True C:\Users\rolfa\DataspellProjects\reddit\reddit-label\data\submissions.ndjson
Exists comments   : True C:\Users\rolfa\DataspellProjects\reddit\reddit-label\data\comments.ndjson


,id,body
0,grd997s,I am firmly convinced this is why the former g...
1,grdh95m,I believe Trump's financial records contain ev...
2,grdjw0s,The Republican Party became the Party of NO wh...
3,grdm7ny,"Roxanna, apparently you have the right forum t..."
4,grdxka2,Guess I\`ll follow you and txgrandpa to this s...
...,...,...
59227,j2fpo68,&gt;Trump derangement syndrome \n&gt; \n&gt;...
59228,j2fqxf9,"&amp;#x200B;\n\n&amp;#x200B;\n\nYou post.. ""Ag..."
59229,j2ftclk,"Yeah, I read the ridiculous definition you hav..."
59230,j2ftjzb,"Oh, I know what it’s about, which is why I kee..."


In [29]:

# ------------------------
# Task schema specifications (for strict validation)
# ------------------------

TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "stance_intensity": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "stance_intensity",
        "score_key": "score",
        "score_type": (int, float, str),   # "ABSTAIN" allowed
        "confidence_range": (0.0, 1.0),
        "score_range": (-1.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "epistemic_modality": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "epistemic_modality",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "justification_density": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "justification_density",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")),  # non-negative float
        "allow_abstain": True,
        "label_key": None,
    },
    "responsiveness": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "responsiveness",
        "score_key": "score",
        "score_type": (int, float, str),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,  # Will ABSTAIN because no parent text is provided
        "label_key": None,
    },
    "civility": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "civility",
        "score_key": None,
        "label_key": "label",
        "label_type": (int, str),  # 1..6 or "ABSTAIN"
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
}

# ------------------------
# System prompt (passed verbatim to the model)
# ------------------------

SYSTEM_PROMPT = """You are a careful, literal discourse annotator. Use ONLY the provided TEXT,
PARENT_TEXT (if given), and TOPIC_DEF. Do not infer beyond the text or
use any external knowledge. If evidence is insufficient, return "ABSTAIN".
Output must be strictly valid JSON matching the registered schema for the task.
Include a "confidence" field in [0,1].
B.2 B2. Instruction templates and schemas
Below we include representative instruction + schema pairs. All tasks follow the same schema:
English only; if ¡15 tokens then ABSTAIN.
Stance intensity (continuous; [−1, 1])
TASK: Estimate the author’s stance toward TOPIC_DEF as a continuous
score in [-1.0, 1.0]. Use explicit valenced statements linked to the topic.
If ambiguous or absent, return "ABSTAIN".
JSON schema:
{
"task": "stance_intensity",
"score": <float in [-1.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Epistemic modality (continuous; [0,1])
TASK: Score epistemic modality: density of hedges, modal verbs, and explicit
uncertainty, normalized by text length. Higher = more expressed humility.
Return "ABSTAIN" if <15 tokens.
JSON schema:
{
"task": "epistemic_modality",
"score": <float in [0.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Justification density (per 100 words)
TASK: Identify distinct justification units (claim + warrant) and report
the count normalized per 100 words. Use span-unitization rules provided.
Return "ABSTAIN" if insufficient evidence.
JSON schema:
{
"task": "justification_density",
"score": <non-negative float or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Responsiveness (continuous; [0,1])
TASK: Rate how directly this reply addresses its PARENT_TEXT’s content
(semantic overlap / engagement). 0 = unrelated; 1 = direct, on-topic response.
Return "ABSTAIN" if PARENT_TEXT not provided.
JSON schema:
{
"task": "responsiveness",
"score": <float in [0.0,1.0] or "ABSTAIN">,
"confidence": <float in [0,1]>
}
Civility (ordinal 1–6)
TASK: Rate civility on a 1 (highly uncivil/insulting) to 6 (highly civil)
scale, focusing on tone, directness, and presence of insults.
Return "ABSTAIN" if ambiguous.
JSON schema:
{
"task": "civility",
"label": <integer 1..6 or "ABSTAIN">,
"confidence": <float in [0,1]>,
"probs": { "1":.., "2":.., ... "6":.. } // optional
}
"""

# Template that tells the model which single task to execute now
TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name}. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)

# ------------------------
# HTTP call utilities
# ------------------------

def call_lmstudio_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.
    Raises requests.RequestException on network/HTTP errors.
    """
    url = f"{LMSTUDIO_BASE_URL}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()

def _is_number(x: Any) -> bool:
    """Helper: check numeric types."""
    return isinstance(x, (int, float))

# ------------------------
# Response validation
# ------------------------

def validate_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate the model's JSON against the task schema.
    Returns None if valid; otherwise returns an error message.
    """
    spec = TASK_SPECS[task]

    # Must be a dict
    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    # Required keys present?
    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    # Exact "task" value
    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    # confidence in [0,1]
    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo, hi = spec["confidence_range"]
    if not (lo <= conf <= hi):
        return f'"confidence" must be in [{lo}, {hi}]'

    # Civility uses an ordinal "label"
    if task == "civility":
        label = obj.get(spec["label_key"])
        if isinstance(label, str):
            if label != "ABSTAIN" and not label.isdigit():
                return '"label" must be integer 1..6 or "ABSTAIN"'
        elif isinstance(label, int):
            if not (1 <= label <= 6):
                return '"label" integer must be in [1,6]'
        else:
            return '"label" must be int or "ABSTAIN"'
        return None

    # All other tasks use a numeric "score" or "ABSTAIN"
    score = obj.get(spec["score_key"])
    if isinstance(score, str):
        if not spec["allow_abstain"] or score != "ABSTAIN":
            return f'"{spec["score_key"]}" must be a number or "ABSTAIN"'
        return None
    elif _is_number(score):
        lo, hi = spec["score_range"]
        if not (lo <= float(score) <= hi):
            return f'"{spec["score_key"]}" out of range [{lo}, {hi}]'
        return None
    else:
        return f'"{spec["score_key"]}" must be number or "ABSTAIN"'

# ------------------------
# Prompt construction
# ------------------------

def build_user_prompt(task: str, text: str) -> str:
    """
    Compose the user message that includes only TEXT and the task directive.
    Note: We do not provide PARENT_TEXT or TOPIC_DEF in this simplified setup.
    """
    parts = {"TEXT": text}
    directive = TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    return json.dumps(parts, ensure_ascii=False) + "\n\n" + directive

# ------------------------
# Single annotation with retries
# ------------------------

def annotate_one(task: str, text: str) -> Dict[str, Any]:
    """
    Run one (task, text) through LM Studio with retries and strict JSON validation.
    Returns the valid task-JSON on success, or {"task": task, "error": "..."} after all retries fail.
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(task, text)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)

            # Expect strictly JSON (no Markdown). Try to parse.
            obj = json.loads(raw)

            # Validate against the schema for this task
            err = validate_response(task, obj)
            if err is None:
                return obj
            else:
                last_err = f"Schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            # Network or HTTP-level error
            last_err = f"HTTP error (attempt {attempt}): {e}"

        except json.JSONDecodeError as e:
            # Model returned non-JSON or malformed JSON
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        # Exponential backoff before retrying
        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    # After exhausting all retries, return an error object
    return {
        "task": task,
        "error": last_err or "Unknown error",
    }

# ------------------------
# NDJSON writer
# ------------------------

def write_ndjson_line(fp, obj: Dict[str, Any]) -> None:
    """Write a single JSON object as one NDJSON line."""
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")

# ------------------------
# Input normalization
# ------------------------

NormalizedItem = Tuple[str, str]  # (comment_id, body_text)

def _normalize_input(
    comments: Union[
        Dict[str, Any],                 # either a single reddit dict with keys 'id' & 'body' OR mapping id->text
        List[Dict[str, Any]],           # list of reddit dicts
        pd.DataFrame,                   # DataFrame with columns ['id','body']
    ]
) -> Iterable[NormalizedItem]:
    """
    Normalize different input shapes to an iterator of (comment_id, body_text).
    Accepted forms:
      - Single Reddit-style dict with 'id' and 'body'
      - List of Reddit-style dicts with 'id' and 'body'
      - Mapping {id: text}
      - pd.DataFrame with columns ['id','body']
    """
    # Case: DataFrame
    if isinstance(comments, pd.DataFrame):
        if not {"id", "body"}.issubset(comments.columns):
            raise ValueError("DataFrame input must contain columns ['id','body'].")
        for _, row in comments.iterrows():
            cid = str(row["id"])
            text = str(row["body"])
            if text and cid:
                yield (cid, text)
        return

    # Case: dict
    if isinstance(comments, dict):
        # Reddit-style single item?
        if "id" in comments and "body" in comments:
            yield (str(comments["id"]), str(comments["body"]))
            return
        # Otherwise assume mapping id -> text
        for k, v in comments.items():
            cid = str(k)
            text = str(v)
            if text and cid:
                yield (cid, text)
        return

    # Case: list of dicts
    if isinstance(comments, list):
        for item in comments:
            if not isinstance(item, dict):
                raise ValueError("List input must contain dictionaries with keys ['id','body'].")
            if "id" not in item or "body" not in item:
                raise ValueError("Each dictionary in the list must have 'id' and 'body'.")
            yield (str(item["id"]), str(item["body"]))
        return

    raise TypeError(
        "Unsupported input type. Provide a DataFrame with ['id','body'], "
        "a dict mapping id->text, a single reddit dict with 'id'/'body', or a list of such dicts."
    )

# ------------------------
# Main pipeline
# ------------------------

def run_pipeline(
        comments: Union[Dict[str, Any], List[Dict[str, Any]], pd.DataFrame],
        tasks: Optional[List[str]] = None,
        ndjson_path: str = "labels.ndjson",
) -> None:
    """
    Run the labeling pipeline.

    Args:
        comments: Input data. Supported:
                  - Single reddit dictionary with keys ['id','body']
                  - List of reddit dictionaries with keys ['id','body']
                  - Dict mapping id->text
                  - pandas DataFrame with columns ['id','body']
        tasks: Optional list of task names; defaults to all implemented tasks.
        ndjson_path: Output file path for NDJSON.

    Output format (NDJSON):
        One line per (comment_id, task), with a JSON object:
        {
          "comment_id": "<id string>",
          "comment_index": <int>,         # running index in this pipeline call (0..N-1)
          "task": "<task_name>",
          "result": { ... }               # validated task JSON or {"task": "<task>", "error": "..."}
        }
    """
    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    iterator = list(_normalize_input(comments))

    with open(ndjson_path, "w", encoding="utf-8") as f:
        for idx, (comment_id, text) in enumerate(iterator):
            for task in tasks:
                result = annotate_one(task, text)
                out = {
                    "comment_id": comment_id,
                    "comment_index": idx,
                    "task": task,
                    "result": result,
                }
                write_ndjson_line(f, out)





In [30]:
import os
from typing import Set, Tuple, Callable, Optional, List, Dict, Any
import pandas as pd
import json

def _load_done_pairs(ndjson_path: str) -> Set[Tuple[str, str]]:
    """
    Liest (comment_id, task)-Pairs aus einer bereits existierenden NDJSON,
    um doppelte Arbeit zu vermeiden (Resume-Funktionalität).
    """
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(ndjson_path):
        return done
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                cid = str(obj.get("comment_id", ""))
                t = str(obj.get("task", ""))
                if cid and t:
                    done.add((cid, t))
            except Exception:
                # kaputte Zeilen ignorieren, weiter
                continue
    return done


def label_dataframe(
    df: pd.DataFrame,
    tasks: Optional[List[str]] = None,
    ndjson_path: str = "labels.ndjson",
    id_col: str = "id",
    text_col: str = "body",
    batch_size: int = 500,
    start_index: int = 0,
    end_index: Optional[int] = None,
    skip_existing: bool = True,
    progress: Optional[Callable[[int, int, Dict[str, Any]], None]] = None,
) -> Dict[str, Any]:
    """
    Labelt Kommentare aus einem DataFrame in Batches und schreibt NDJSON Zeile-für-Zeile.

    Parameter
    ---------
    df : pd.DataFrame
        DataFrame mit mindestens den Spalten `id_col` und `text_col`.
    tasks : list[str] | None
        Liste der Tasks; None => alle TASK_SPECS.
    ndjson_path : str
        Ausgabedatei (append-sicher; erstellt sie, falls nicht vorhanden).
    id_col : str
        Spaltenname für die Kommentar-ID.
    text_col : str
        Spaltenname für den Kommentar-Text (body).
    batch_size : int
        Anzahl Zeilen pro Batch (nur Speicherkontrolle/Flush).
    start_index : int
        Startzeilenindex (inklusive) im DataFrame.
    end_index : int | None
        Endzeilenindex (exklusive); None => bis zum Ende.
    skip_existing : bool
        Falls True, werden bereits vorhandene (comment_id, task) aus `ndjson_path` übersprungen.
    progress : callable | None
        Optionaler Callback `progress(done_rows, total_rows, stats_dict)`.

    Rückgabe
    --------
    dict :
        Zusammenfassung mit Countern (processed_rows, written_records, skipped_records, errors).

    Hinweise
    --------
    - Erfordert die Funktionen `annotate_one` und `write_ndjson_line`
    - NDJSON wird fortlaufend geschrieben (kein Memory-Buffer nötig).
    - Robust gegen NaN/leerem Text; solche Zeilen werden übersprungen.
    """
    assert id_col in df.columns, f"Spalte '{id_col}' fehlt im DataFrame."
    assert text_col in df.columns, f"Spalte '{text_col}' fehlt im DataFrame."

    if tasks is None:
        # Alle registrierten Tasks verwenden
        tasks = list(TASK_SPECS.keys())

    # Slicing des DataFrames
    n_total = len(df)
    if end_index is None or end_index > n_total:
        end_index = n_total
    if start_index < 0:
        start_index = 0
    if start_index >= end_index:
        return {
            "processed_rows": 0,
            "written_records": 0,
            "skipped_records": 0,
            "errors": 0,
            "note": "Nichts zu verarbeiten (Start >= Ende).",
        }

    # Resume: bereits gelabelte (id, task) einlesen
    already_done: Set[Tuple[str, str]] = set()
    if skip_existing:
        already_done = _load_done_pairs(ndjson_path)

    # Datei im Append-Modus öffnen (robust gegenüber Abbrüchen)
    written_records = 0
    skipped_records = 0
    error_count = 0
    processed_rows = 0

    with open(ndjson_path, "a", encoding="utf-8") as fp:
        # Batch-weise iterieren
        for batch_start in range(start_index, end_index, batch_size):
            batch_end = min(batch_start + batch_size, end_index)
            batch = df.iloc[batch_start:batch_end]

            for local_idx, row in batch.iterrows():
                comment_id = str(row[id_col]) if pd.notna(row[id_col]) else ""
                text = str(row[text_col]) if pd.notna(row[text_col]) else ""

                if not comment_id or not text or not text.strip():
                    skipped_records += len(tasks)  # nichts zu tun für alle tasks dieser Zeile
                    processed_rows += 1
                    if progress:
                        progress(processed_rows, end_index - start_index, {
                            "written_records": written_records,
                            "skipped_records": skipped_records,
                            "errors": error_count
                        })
                    continue

                for task in tasks:
                    if skip_existing and (comment_id, task) in already_done:
                        skipped_records += 1
                        continue

                    try:
                        result = annotate_one(task, text)
                        out = {
                            "comment_id": comment_id,
                            "comment_index": int(local_idx),  # Original-Index im DF
                            "task": task,
                            "result": result,
                        }
                        write_ndjson_line(fp, out)
                        written_records += 1
                        # Sofort flushen, um Datenverlust bei Abbruch zu minimieren
                        if written_records % 100 == 0:
                            fp.flush()
                    except Exception:
                        error_count += 1
                        # wir loggen keine Tracebacks hier; optional nachrüsten

                processed_rows += 1

                if progress:
                    progress(processed_rows, end_index - start_index, {
                        "written_records": written_records,
                        "skipped_records": skipped_records,
                        "errors": error_count
                    })

    return {
        "processed_rows": processed_rows,
        "written_records": written_records,
        "skipped_records": skipped_records,
        "errors": error_count,
    }


In [ ]:
# Anzahl der zu labelnden Kommentare selbst festlegen
n_entries = 1000   # <== hier beliebig anpassen (z. B. 10, 500, 20000, len(df_comments))

# gewünschte Tasks
tasks = [
    "stance_intensity",
    "civility",
    "epistemic_modality",
    "justification_density",
    "responsiveness",
]

# Teil-DataFrame erstellen (erste n_entries Zeilen)
df_subset = df_comments.iloc[:n_entries]

# Fortschrittsanzeige (optional)
def simple_progress(done, total, s):
    if done % 500 == 0 or done == total:
        print(f"[{done}/{total}] written={s['written_records']} skipped={s['skipped_records']} errors={s['errors']}")

# Labeln starten
stats = label_dataframe(
    df=df_subset,
    tasks=tasks,
    ndjson_path=f"labels_{n_entries}.ndjson",  # Ausgabe-Datei z. B. labels_1000.ndjson
    batch_size=1000,
    skip_existing=True,
    progress=simple_progress,
)

print(f"Labeling abgeschlossen ({n_entries} Einträge).")
print(stats)

